# Model Store

In [1]:
import boto3
import os

bucket = "sagemaker-us-east-1-318401170150"
prefix = "buoy/xgboost-training"

model_local_path = "models/buoy_wave_model.joblib"
model_s3_path = f"s3://{bucket}/{prefix}/buoy_wave_model.joblib"

s3 = boto3.client("s3")
s3.upload_file(model_local_path, bucket, f"{prefix}/buoy_wave_model.joblib")

print(f"Uploaded model to {model_s3_path}")

Uploaded model to s3://sagemaker-us-east-1-318401170150/buoy/xgboost-training/buoy_wave_model.joblib


In [4]:
import boto3
from time import gmtime, strftime

sm = boto3.client("sagemaker", region_name="us-east-1")

role = "arn:aws:iam::318401170150:role/LabRole"

# Use latest prebuilt SKLearn container
region = "us-east-1"
sklearn_image = f"683313688378.dkr.ecr.{region}.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"

model_name = "buoy-wave-model-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())

create_model_response = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": sklearn_image,
        "ModelDataUrl": model_s3_path,
        "Environment": {}  # Optional environment variables
    }
)

print("Created model:", create_model_response["ModelArn"])

Created model: arn:aws:sagemaker:us-east-1:318401170150:model/buoy-wave-model-2026-02-13-23-43-07


In [6]:
endpoint_config_name = "buoy-wave-endpoint-config-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())

endpoint_config_response = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": "ml.m5.xlarge"
    }]
)

print("Created endpoint config:", endpoint_config_response["EndpointConfigArn"])

Created endpoint config: arn:aws:sagemaker:us-east-1:318401170150:endpoint-config/buoy-wave-endpoint-config-2026-02-13-23-43-28


In [7]:
endpoint_name = "buoy-wave-endpoint-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())

create_endpoint_response = sm.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name
)

print("Endpoint creation started...")

import time

while True:
    status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
    print(f"Endpoint status: {status}")
    if status == "InService":
        break
    elif status in ["Failed", "RollingBack"]:
        raise RuntimeError("Endpoint creation failed")
    time.sleep(30)

print(f"Endpoint {endpoint_name} is ready!")

Endpoint creation started...
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Creating
Endpoint status: Failed


RuntimeError: Endpoint creation failed

In [ ]:
import pandas as pd

sm_runtime = boto3.client("sagemaker-runtime", region_name="us-east-1")

# Example input CSV
test_data = pd.read_csv("test_data.csv")

response = sm_runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="text/csv",
    Body=test_data.to_csv(header=False, index=False)
)

predictions = response['Body'].read().decode('utf-8')
print("Predictions:", predictions)

In [ ]:
import json

model_card = {
    "Model Name": model_name,
    "S3 Model Path": model_s3_uri,
    "Features": list(test_data.columns),
    "Target": "E_star",
    "Algorithm": "HistGradientBoosting / sklearn",
    "Endpoint Name": endpoint_name,
    "Description": "Locally trained buoy wave prediction model deployed on SageMaker",
}

with open("buoy_wave_model_card.json", "w") as f:
    json.dump(model_card, f, indent=4)

print("Saved model card: buoy_wave_model_card.json")

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>